# BARC-SS example: HUC8 11110203, trapezoidal channel a = 0.4

OWP HAND-FIM rating curves miss the river channel below the water surface because the DEM does not capture it. This notebook adds that channel to the hydrotables with `barc_ss.py`, then runs HAND-FIM with [FIMserv](https://github.com/sdmlua/FIMserv):

- **BARC**: adds the bankfull channel area and wetted perimeter to the hydrotable and recomputes discharge with Manning's equation.
- **BARC-SS**: BARC, plus shifts each rating curve's stage up by the reach's mean-flow depth and anchors it at zero flow.

Example: **HUC8 11110203** (Arkansas River), **trapezoidal channel with a = 0.4**, flood of **3 January 2016, 00:00**.

FIMserv writes `code/`, `data/`, and `output/` folders next to this notebook.

### STEP 1. Install and import

Install once:
```bash
pip install fimserve "git+https://github.com/Reizrb/HydroGeom"
```

In [ ]:
import sys
sys.path.append("..")          # barc_ss.py is in the repository's main folder

import fimserve as fm
import matplotlib.pyplot as plt
import barc_ss

### STEP 2. Settings

- `method`: `"BARC"` or `"BARC-SS"`
- `shape` and `value`: channel shape used to compute bankfull area and wetted perimeter
  - `"a"`: trapezoid, `value` = bottom width / top width (0 = triangle, 1 = rectangle)
  - `"r"`: Dingman's r, `value` = r (1 = triangle, 2 = parabola, larger = rectangle)
  - `"z"`: side slope, `value` = z (z:1)

In [ ]:
huc = "11110203"
method = "BARC-SS"
shape, value = "a", 0.4

start_date = "2015-12-26"
end_date = "2016-01-05"
value_times = ["2016-01-03 00:00:00"]

flood_dir = f"output/flood_{huc}"

### STEP 3. Download OWP HAND-FIM (version 4.5) with FIMserv

In [ ]:
fm.DownloadHUC8(huc, version="4.5")

### STEP 4. Channel geometry for every reach

Gets bankfull width and depth and mean-flow depth from [HydroGeom](https://github.com/Reizrb/HydroGeom) for every reach in the HUC8, and computes bankfull area (`bnk_xsce_A`) and wetted perimeter (`bnk_wet_p`) for the chosen shape.

In [ ]:
ids = barc_ss.feature_ids(flood_dir, huc)
geometry = barc_ss.channel_geometry(ids, shape=shape, value=value)
geometry.head()

### STEP 5. Apply BARC or BARC-SS to the hydrotables

The hydrotables are rewritten in place. The originals are saved in `output/flood_11110203/original_hydrotables/` the first time, and every run starts from them.

In [ ]:
barc_ss.apply_barc(flood_dir, huc, geometry, method=method)

### STEP 6. Rating curve before and after

In [ ]:
barc_ss.plot_rating_curve(flood_dir, huc, label=method)    # add hydroid=... for a specific reach
plt.show()

### STEP 7. NWM discharge and HAND-FIM

In [ ]:
fm.getNWMretrospectivedata(huc, start_date, end_date, value_times=value_times)
fm.runOWPHANDFIM(huc)

### STEP 8. Map the flooded area

In [ ]:
import glob
import geopandas as gpd
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap

fim = sorted(glob.glob(f"{flood_dir}/{huc}_inundation/*_inundation.tif"))[-1]
with rasterio.open(fim) as r:
    flooded = r.read(1) > 0
    b, crs, res = r.bounds, r.crs, r.res

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(np.ma.masked_where(~flooded, flooded), cmap=ListedColormap(["#1f78b4"]),
          extent=(b.left, b.right, b.bottom, b.top), interpolation="nearest")
gpd.read_file(f"{flood_dir}/{huc}/wbd8_clp.gpkg").to_crs(crs).boundary.plot(ax=ax, color="0.4", lw=0.8)
ax.set_title(f"{method} ({shape} = {value}), HUC8 {huc}: {flooded.sum() * res[0] * res[1] / 1e6:,.1f} km² flooded")
ax.set_axis_off()
plt.savefig(f"FIM_{huc}_{method}_{shape}{value}.png", dpi=200, bbox_inches="tight")
plt.show()

### Run again with another method or shape

FIMserv skips HAND-FIM if a flood map for the same discharge already exists, so rename the current map first. Then change the settings in STEP 2 and run STEPS 4 to 8 again (no new download needed).

In [ ]:
import os
os.rename(fim, fim.replace("_inundation.tif", f"_{method}_{shape}{value}_inundation.tif"))

To map the original HAND-FIM for comparison, run `barc_ss.restore_hydrotables(flood_dir)` instead of STEPS 4 and 5.